# Train Baseline (TF-IDF) trên ViFactCheck
Notebook này dùng để load dataset ViFactCheck, chuẩn bị dữ liệu (chia train/test) và huấn luyện các mô hình cơ bản (Baseline) dùng TF-IDF.
Các mô hình truyền thống sử dụng: Logistic Regression, SVM, Naive Bayes.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Thêm root folder vào sys.path để có thể import config
sys.path.insert(0, str(Path().resolve().parent))
from config import VIFACTCHECK_DIR

In [2]:
# 1. Load Data
csv_path = VIFACTCHECK_DIR / "vifactcheck_full.csv"
if not csv_path.exists():
    print(f"⚠️ Không tìm thấy file: {csv_path}")
    print("==> Hãy chạy lệnh `python dataset/download_datasets.py` trước để download dữ liệu.")
else:
    df = pd.read_csv(csv_path)
    print(f"Load thành công {len(df)} samples từ {csv_path}")
    display(df.head())

Load thành công 7232 samples từ D:\Work\project UET\fake-new-detection\dataset\raw\vifactcheck\vifactcheck_full.csv


,text,evidence,label,original_label,source,split
0,"Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ,...","Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó T...",real,SUPPORTED,vifactcheck,train
1,Hành vi của Tô Văn Hải là cho phép người khác ...,Tô Văn Hải đã có hành vi cho phép người khác đ...,real,SUPPORTED,vifactcheck,train
2,SAWACO thông báo tạm ngưng cung cấp nước để th...,SAWACO thông báo tạm ngưng cung cấp nước để th...,fake,REFUTED,vifactcheck,train
3,"CLB luôn chuẩn bị rất kỹ lưỡng, chỉn chu chươn...",Là chương trình lớn nhất của CLB trong năm nên...,suspicious,NOT ENOUGH INFO,vifactcheck,train
4,"ILA tiếp nhận và hỗ trợ học sinh miễn phí, Bé ...","Bé được tham gia kiểm tra trình độ đầu vào, tư...",suspicious,NOT ENOUGH INFO,vifactcheck,train


ViFactCheck gồm các cột đáng chú ý: `text` (claim), `evidence`, `label` (real/fake/suspicious), `original_label`, `source`, `split`.
Vì bộ dữ liệu đã được chia sẵn tập train/dev/test (thể hiện ở cột `split`), ta sẽ tận dụng cột này luôn.

In [3]:
# Lọc bỏ các dòng bị thiếu claim hoặc label
df = df.dropna(subset=["text", "label"])

# Chia Train / Val / Test
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'dev'].copy()
test_df = df[df['split'] == 'test'].copy()

print("Train sizes:", len(train_df))
print("Val sizes:", len(val_df))
print("Test sizes:", len(test_df))

Train sizes: 5062
Val sizes: 723
Test sizes: 1447


In [4]:
X_train = train_df['text'].tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].tolist()
y_test = test_df['label'].tolist()

print("Label distribution (Train):", train_df['label'].value_counts().to_dict())

Label distribution (Train): {'real': 1751, 'fake': 1658, 'suspicious': 1653}


### Trích xuất đặc trưng với TF-IDF
Sử dụng thư viện `underthesea` (nếu đã cài đặt) để thực hiện word segmentation chuẩn tiếng Việt. Nếu máy tính bạn chưa có thì TF-IDF sẽ tự động fallback sang cơ chế tách chữ bằng các dấu khoảng trắng thông thường.

In [5]:
try:
    from underthesea import word_tokenize
    def vi_tokenizer(text):
        return word_tokenize(text, format="text").split()
except ImportError:
    print("underthesea chưa cài, dùng default tokenizer (split theo space).")
    def vi_tokenizer(text):
        return text.split()

# Sinh đặc trưng (features) TF-IDF (bỏ qua những từ hiếm xuất hiện: min_df=2)
vectorizer = TfidfVectorizer(
    tokenizer=vi_tokenizer,
    ngram_range=(1, 2), # Cả unigram và bigram
    max_features=10000,
    min_df=2
)

print("Đang tính weight TF-IDF...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Kích thước từ vựng (vocabulary):", len(vectorizer.vocabulary_))

Đang tính weight TF-IDF...


d:\Work\project UET\fake-new-detection\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Kích thước từ vựng (vocabulary): 10000


In [6]:
# Định nghĩa danh sách các thuật toán phân lớp cổ điển (Machine Learning)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Linear SVM": LinearSVC(max_iter=2000, class_weight="balanced", random_state=42),
    "Naive Bayes": MultinomialNB()
}

print("Bắt đầu huấn luyện mô hình:\n")
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    
    # Đánh giá trên Validation Set
    y_pred_val = model.predict(X_val_tfidf)
    f1 = f1_score(y_val, y_pred_val, average="macro")
    print(f"[{name}] Validation F1 (Macro): {f1:.4f}")

Bắt đầu huấn luyện mô hình:

[Logistic Regression] Validation F1 (Macro): 0.4123
[Linear SVM] Validation F1 (Macro): 0.4192
[Naive Bayes] Validation F1 (Macro): 0.4045


In [7]:
best_model_name = "Linear SVM" # Tuỳ chỉnh Model nào có Validation F1 cao nhất
best_model = models[best_model_name]

print(f"ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT ({best_model_name}) TRÊN TEST SET:")
y_pred_test = best_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_test))

ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT (Linear SVM) TRÊN TEST SET:
              precision    recall  f1-score   support

        fake       0.39      0.35      0.37       468
        real       0.45      0.47      0.46       508
  suspicious       0.41      0.43      0.42       471

    accuracy                           0.42      1447
   macro avg       0.42      0.42      0.42      1447
weighted avg       0.42      0.42      0.42      1447

